# Берем готовый датасет lstm и с помощью bigquery собираем датасет для gnn

In [4]:
# Установить клиентскую библиотеку
%pip uninstall -y pandas
# свежий pip подтянет правильный wheel
%pip install --upgrade pip           
%pip install "pandas>=2.2.0"

%pip install google-cloud-bigquery


Found existing installation: pandas 2.2.3
Uninstalling pandas-2.2.3:
  Successfully uninstalled pandas-2.2.3
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached pandas-2.2.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
Using cached pandas-2.2.3-cp312-cp312-win_amd64.whl (11.5 MB)
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [6]:
# Шаг 1: Авторизация (если не делал)
!gcloud auth application-default login --quiet

# Шаг 2: Установка проекта для квоты
!gcloud auth application-default set-quota-project celtic-tendril-459507-q8

"gcloud" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.
"gcloud" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


In [30]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "C:\\Users\\fesevu\\AppData\\Roaming\\gcloud\\application_default_credentials.json"
# замените на свой ID
os.environ["GOOGLE_CLOUD_PROJECT"] = "celtic-tendril-459507-q8"

# Разделяем на контракты и аддреса

In [6]:
import pandas as pd

# Загрузка исходного CSV
df = pd.read_csv('./data2/merged_6_dataset.csv')

# Удаление строк с suspect
df = df[df['FLAG'] != 'suspect']

# Замена значений FLAG
df['FLAG'] = df['FLAG'].map({'legit': 0, 'scam': 1})

# Приведение is_contract к булевому типу
df['is_contract'] = df['is_contract'].astype(bool)

# Разделение на два датасета
df_contracts = df[df['is_contract'] == True].drop(columns=['is_contract'])
df_non_contracts = df[df['is_contract'] == False].drop(columns=['is_contract'])

# Сохранение в отдельные файлы
df_contracts.to_csv('./data/contracts_only.csv', index=False)
df_non_contracts.to_csv('./data/non_contracts_only.csv', index=False)


In [ ]:
import pandas as pd

# Загрузка датасета с контрактами
df = pd.read_csv('./data/contracts_only.csv')

# Отбор всех флагов 1
df_flag_1 = df[df['FLAG'] == 1]
print(df_flag_1.shape)

# Сколько нужно флагов 0, чтобы получить ровно 3100 строк
needed_flag_0 = 3164 - len(df_flag_1)

# Случайная выборка флагов 0
df_flag_0 = df[df['FLAG'] == 0].sample(n=needed_flag_0, random_state=42)

# Объединение
df_sampled = pd.concat([df_flag_1, df_flag_0]).sample(frac=1, random_state=42)  # перемешаем

# Сохранение
df_sampled.to_csv('./data/contracts_sampled.csv', index=False)


(824, 3)


# Загружает LSTM датасет


In [7]:
import pandas as pd
from google.cloud import bigquery

# 1. Считываем LSTM-датасет
lstm_df = pd.read_csv('./data/transaction_dataset.csv', usecols=['Address','FLAG'])
lstm_df.columns = ['id','label']
seed_addrs = set(lstm_df['id'])

# Новая версия

In [10]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Генерирует SQL для «ограниченного 2-hop BFS» + dry-run оценку.
"""
import pandas as pd
import subprocess
import shlex
import json
import textwrap

# ======= ПОДСТАВЬТЕ СВОЁ ======================================
PROJECT = "celtic-tendril-459507-q8"         # GCP-проект
DATASET = "fraud_gnn"          # datataset в BigQuery
LSTM_CSV = "./data/contracts_sampled.csv"          # исходник
DATE_FROM = "2020-01-01"        # начало окна
DATE_TO = "2023-01-01"        # конец окна
MAX_HOP2 = 50             # лимит узлов hop2
# ===============================================================

# 0) выгружаем LSTM-адреса в CSV для загрузки в BQ
df = pd.read_csv(LSTM_CSV, usecols=["Address", "FLAG"])
df["Address"] = df["Address"].str.strip().str.lower()
df.to_csv("./bq/lstm_addresses.csv", index=False)

print(f"→ Загрузите ./bq/lstm_addresses.csv в `{PROJECT}.{DATASET}.lstm_addresses`"
      " (Address STRING, FLAG INT64)")

→ Загрузите ./bq/lstm_addresses.csv в `celtic-tendril-459507-q8.fraud_gnn.lstm_addresses` (Address STRING, FLAG INT64)


In [12]:
# --------------- динамически собираем SQL ---------------------
sql = f"""
-- Параметры
DECLARE date_from DATE   DEFAULT '{DATE_FROM}';
DECLARE date_to   DATE   DEFAULT '{DATE_TO}';
DECLARE max_hop2  INT64  DEFAULT {MAX_HOP2};

-- 0. Базовая таблица LSTM
CREATE OR REPLACE TABLE `{PROJECT}.{DATASET}.lstm` AS
SELECT
  LOWER(Address)   AS addr,
  CAST(FLAG AS BOOL) AS is_scam
FROM `{PROJECT}.{DATASET}.lstm_addresses`;

-- 1. Периоды активности seed-аккаунтов
CREATE OR REPLACE TABLE `{PROJECT}.{DATASET}.account_activity` AS
SELECT
  id,
  MIN(DATE(block_timestamp)) AS first_seen,
  MAX(DATE(block_timestamp)) AS last_seen
FROM (
  SELECT LOWER(from_address) AS id, block_timestamp
    FROM `bigquery-public-data.crypto_ethereum.transactions`
  UNION ALL
  SELECT LOWER(to_address)   AS id, block_timestamp
    FROM `bigquery-public-data.crypto_ethereum.transactions`
)
WHERE id IN (SELECT addr FROM `{PROJECT}.{DATASET}.lstm`)
GROUP BY id;

-- 2. Hop-1 по каждому seed-аккаунту
CREATE OR REPLACE TABLE `{PROJECT}.{DATASET}.hop1` AS
WITH seeds AS (
  SELECT l.addr   AS seed,
         a.first_seen,
         a.last_seen,
         l.is_scam
  FROM `{PROJECT}.{DATASET}.lstm`  AS l
  JOIN `{PROJECT}.{DATASET}.account_activity` AS a
    ON l.addr = a.id
)
SELECT DISTINCT
  s.seed,
  LOWER(
    IF(t.from_address = s.seed, t.to_address, t.from_address)
  ) AS hop1_addr,
  s.is_scam
FROM seeds AS s
JOIN `bigquery-public-data.crypto_ethereum.transactions` AS t
  ON DATE(t.block_timestamp) BETWEEN s.first_seen AND s.last_seen
 AND t.value > 0
 AND (
       LOWER(t.from_address) = s.seed
    OR LOWER(t.to_address)   = s.seed
     );

-- 3. Hop-2 с лимитом per seed: безлимит для scam, до 50 для legit
CREATE OR REPLACE TABLE `{PROJECT}.{DATASET}.hop2` AS
WITH ranked AS (
  SELECT
    h1.seed,
    LOWER(
      IF(t.from_address = h1.hop1_addr, t.to_address, t.from_address)
    ) AS hop2_addr,
    h1.is_scam         AS is_scam_seed,
    COUNT(*)           AS tx_cnt
  FROM `{PROJECT}.{DATASET}.hop1`       AS h1
  JOIN `bigquery-public-data.crypto_ethereum.transactions` AS t
    ON t.value > 0
   AND (
         LOWER(t.from_address) = h1.hop1_addr
      OR LOWER(t.to_address)   = h1.hop1_addr
       )
  GROUP BY h1.seed, hop2_addr, h1.is_scam
)
SELECT seed, hop2_addr AS addr
FROM ranked
QUALIFY
  ROW_NUMBER() OVER (
    PARTITION BY seed
    ORDER BY tx_cnt DESC
  ) <= CASE WHEN is_scam_seed THEN 99999999 ELSE 50 END;

-- 4. Собираем итоговый список узлов
CREATE OR REPLACE TABLE `{PROJECT}.{DATASET}.gnn_accounts` AS
SELECT
  addr AS id,
  ANY_VALUE(is_scam) AS is_scam
FROM (
  SELECT addr, is_scam       FROM `{PROJECT}.{DATASET}.lstm`
  UNION DISTINCT
  SELECT hop1_addr, NULL     FROM `{PROJECT}.{DATASET}.hop1`
  UNION DISTINCT
  SELECT addr,      NULL     FROM `{PROJECT}.{DATASET}.hop2`
)
GROUP BY id;

-- 5. Все рёбра между узлами (с тем же partitioning)
CREATE OR REPLACE TABLE `{PROJECT}.{DATASET}.gnn_transactions`
PARTITION BY DATE(block_timestamp)
AS
SELECT
  LOWER(t.from_address) AS src,
  LOWER(t.to_address)   AS dst,
  SAFE_DIVIDE(CAST(t.value AS FLOAT64), 1e18) AS amount,
  t.block_timestamp
FROM `bigquery-public-data.crypto_ethereum.transactions` AS t
JOIN `{PROJECT}.{DATASET}.gnn_accounts` AS a1
  ON LOWER(t.from_address) = a1.id
JOIN `{PROJECT}.{DATASET}.gnn_accounts` AS a2
  ON LOWER(t.to_address)   = a2.id
WHERE t.value > 0
  AND DATE(t.block_timestamp) BETWEEN date_from AND date_to;

-- 6. Стратифицированная выборка ≤150 000 транзакций
CREATE OR REPLACE TABLE `{PROJECT}.{DATASET}.gnn_transactions_sample` AS
WITH
  all_tx AS (
    SELECT * FROM `{PROJECT}.{DATASET}.gnn_transactions`
  ),
  scam_edges AS (
    SELECT t.*
    FROM all_tx AS t
    JOIN `{PROJECT}.{DATASET}.gnn_accounts` AS a1
      ON t.src = a1.id
    JOIN `{PROJECT}.{DATASET}.gnn_accounts` AS a2
      ON t.dst = a2.id
    WHERE a1.is_scam OR a2.is_scam
  ),
  legit_edges AS (
    SELECT t.*
    FROM all_tx AS t
    JOIN `{PROJECT}.{DATASET}.gnn_accounts` AS a1
      ON t.src = a1.id
    JOIN `{PROJECT}.{DATASET}.gnn_accounts` AS a2
      ON t.dst = a2.id
    WHERE NOT a1.is_scam AND NOT a2.is_scam
  ),
  scam_count AS (
    SELECT COUNT(*) AS cnt FROM scam_edges
  ),
  legit_sample AS (
    SELECT *
    FROM legit_edges
    ORDER BY RAND()
    LIMIT GREATEST(150000 - (SELECT cnt FROM scam_count), 0)
  )
SELECT * FROM scam_edges
UNION ALL
SELECT * FROM legit_sample;
"""

# ---- сохраняем и сразу делаем dry-run ------------------------
with open("./bq/create_stratified_bfs.sql", "w", encoding="utf-8") as f:
    f.write(sql)

print("→ SQL для stratified BFS сохранён в ./bq/create_stratified_bfs.sql")

→ SQL для stratified BFS сохранён в ./bq/create_stratified_bfs.sql


In [37]:
from google.cloud import bigquery

# Укажи ID проекта GCP
PROJECT = "celtic-tendril-459507-q8"

# Инициализация клиента
client = bigquery.Client(project=PROJECT)

# Чтение SQL-запроса из файла
# можно также использовать pathlib
sql_path = Path("D:/Fraud/dataset/create_mini_bfs.sql")
sql = sql_path.read_text()

# Конфигурация на dry-run
job_config = bigquery.QueryJobConfig(
    dry_run=True,
    use_query_cache=False
)

# Выполнение dry-run
query_job = client.query(sql, job_config=job_config)

# Вывод информации
print("→ Делаем dry-run…")
print("   totalBytesProcessed = {:.3f} GB".format(
    query_job.total_bytes_processed / 1e9))
if query_job.total_bytes_processed <= 5e9:
    print("   ✅ Можно запускать без риска — не превышает 5 GB.")
else:
    print("   ⚠️ Превышает 5 GB — может потребоваться включённый billing.")

Forbidden: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/celtic-tendril-459507-q8/jobs?prettyPrint=false: <!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 403 (Forbidden)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/branding/googlelogo/1x/googlelogo_color_150x54dp.png) no-repeat;margin-left:-5px}@media only screen and (min-resolution:192dpi){#logo{background:url(//www.goog

# Собираем в цельный датасет

In [6]:
import gzip
import glob
from pathlib import Path
import shutil

# ─── настройка ────────────────────────────────────────────────────────────────
PATTERN = "./data/gnn_transactions-*.csv.gz"     # шаблон шардов
OUT_GZ = "./data/gnn_transactions_full.csv.gz"  # итоговый gzip
NEED_CSV = True                                   # True → сделать и .csv
# ──────────────────────────────────────────────────────────────────────────────

files = sorted(glob.glob(PATTERN))
if not files:
    raise FileNotFoundError(f"Не найдено файлов по шаблону: {PATTERN}")

out_path = Path(OUT_GZ)
out_path.parent.mkdir(parents=True, exist_ok=True)

header_written = False
line_cnt = 0

with gzip.open(out_path, "wt", encoding="utf-8", newline="") as fout:
    for idx, fname in enumerate(files, 1):
        with gzip.open(fname, "rt", encoding="utf-8", newline="") as fin:
            for i, line in enumerate(fin):
                # пропускаем заголовок во всех, кроме первого шарда
                if i == 0 and header_written:
                    continue
                if i == 0 and not header_written:
                    header_written = True
                fout.write(line)
                line_cnt += 1
        print(f"✓ [{idx:>3}/{len(files)}] merged {Path(fname).name}")

print(f"\nСоздан архив → {out_path}   ({line_cnt:,} строк)")

# ─── опционально: распаковать в CSV ───────────────────────────────────────────
if NEED_CSV:
    OUT_CSV = out_path.with_suffix("")   # .csv
    with gzip.open(out_path, "rb") as gzf, OUT_CSV.open("wb") as csvf:
        shutil.copyfileobj(gzf, csvf)
    print(f"Распакован в → {OUT_CSV}")

✓ [  1/851] merged gnn_transactions-000000000000.csv.gz
✓ [  2/851] merged gnn_transactions-000000000001.csv.gz
✓ [  3/851] merged gnn_transactions-000000000002.csv.gz
✓ [  4/851] merged gnn_transactions-000000000003.csv.gz
✓ [  5/851] merged gnn_transactions-000000000004.csv.gz
✓ [  6/851] merged gnn_transactions-000000000005.csv.gz
✓ [  7/851] merged gnn_transactions-000000000006.csv.gz
✓ [  8/851] merged gnn_transactions-000000000007.csv.gz
✓ [  9/851] merged gnn_transactions-000000000008.csv.gz
✓ [ 10/851] merged gnn_transactions-000000000009.csv.gz
✓ [ 11/851] merged gnn_transactions-000000000010.csv.gz
✓ [ 12/851] merged gnn_transactions-000000000011.csv.gz
✓ [ 13/851] merged gnn_transactions-000000000012.csv.gz
✓ [ 14/851] merged gnn_transactions-000000000013.csv.gz
✓ [ 15/851] merged gnn_transactions-000000000014.csv.gz
✓ [ 16/851] merged gnn_transactions-000000000015.csv.gz
✓ [ 17/851] merged gnn_transactions-000000000016.csv.gz
✓ [ 18/851] merged gnn_transactions-000000000017

In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
inspect_gnn_dataset.py
~~~~~~~~~~~~~~~~~~~~~~
Сводит основные метрики датасета:
  • число узлов, размеченных / fraud-узлов
  • число рёбер, диапазон дат
  • min / max / mean / median суммы перевода
  • 5 крупнейших транзакций
  • топ-5 самых «беспокойных» адресов (по степени)
"""

import pandas as pd
import numpy as np
from collections import Counter
from pathlib import Path
from datetime import datetime

# --- настроить под свои имена файлов --------------------------
ACCOUNTS_CSV = Path("./data/gnn_accounts.csv")               # ~ KB-МB
TX_CSV       = Path("./data/gnn_transactions_full.csv.gz")   # или .csv
CHUNK_ROWS   = 1_000_000       # размер чанка при чтении транзакций
SAMPLE_AMT   = 50_000          # сколько сумм берём для оценки медианы
# --------------------------------------------------------------

assert ACCOUNTS_CSV.exists(), f"{ACCOUNTS_CSV} not found"
assert TX_CSV.exists(), f"{TX_CSV} not found"

# 1) узлы -------------------------------------------------------
acc_df = pd.read_csv(
    ACCOUNTS_CSV,
    dtype={"id": "string"},
)

n_nodes      = len(acc_df)
n_labeled    = acc_df["label"].notna().sum()
n_fraud      = int(acc_df["label"].sum(skipna=True))
n_unlabeled  = n_nodes - n_labeled

# 2) рёбра ------------------------------------------------------
dtypes_tx = {"src": "string",
             "dst": "string",
             "amount": "float64"}     # ETH
parse_dates = ["block_timestamp"]

edge_cnt          = 0
amt_sum           = 0.0
amt_min, amt_max  = np.inf, -np.inf
first_ts, last_ts = None, None
top_tx            = []                # 5 крупнейших переводов
deg_counter       = Counter()
amt_samples       = []

for chunk in pd.read_csv(TX_CSV,
                         dtype=dtypes_tx,
                         parse_dates=parse_dates,
                         chunksize=CHUNK_ROWS,
                         compression="infer"):

    edge_cnt += len(chunk)

    # диапазон дат
    ts_min, ts_max = chunk["block_timestamp"].min(
    ), chunk["block_timestamp"].max()
    first_ts = ts_min if first_ts is None else min(first_ts, ts_min)
    last_ts  = ts_max if last_ts  is None else max(last_ts,  ts_max)

    # суммы
    c_amt = chunk["amount"]
    amt_sum  += c_amt.sum()
    amt_min   = min(amt_min, c_amt.min())
    amt_max   = max(amt_max, c_amt.max())

    # случайные сэмплы для оценки медианы
    if len(amt_samples) < SAMPLE_AMT:
        need = SAMPLE_AMT - len(amt_samples)
        amt_samples.extend(c_amt.sample(min(need, len(chunk)),
                                        random_state=42).tolist())

    # топ-5 крупнейших переводов
    top_tx.extend(chunk.nlargest(5, "amount").to_dict("records"))
    top_tx  = sorted(top_tx, key=lambda r: r["amount"], reverse=True)[:5]

    # степени узлов
    deg_counter.update(chunk["src"])
    deg_counter.update(chunk["dst"])

# агрегаты
amt_mean    = amt_sum / edge_cnt
amt_median  = np.median(amt_samples)  # приближённо
top_degree  = deg_counter.most_common(5)

# 3) печатаем ---------------------------------------------------
print("\n=====  GNN DATASET SUMMARY  =====\n")
print(f"Accounts (nodes):        {n_nodes:,}")
print(f"  ├─ labeled:            {n_labeled:,}")
print(f"  ├─   └─ fraud (1):     {n_fraud:,}")
print(f"  └─ unlabeled (NULL):   {n_unlabeled:,}")

print(f"\nTransactions (edges):    {edge_cnt:,}")
print(f"  Time span:             {first_ts:%Y-%m-%d}  →  {last_ts:%Y-%m-%d}")
print(f"  Amount, ETH:")
print(f"    min  = {amt_min:,.6f}")
print(f"    max  = {amt_max:,.6f}")
print(f"    mean = {amt_mean:,.6f}")
print(f"    med. ≈ {amt_median:,.6f}")

print("\nTop-5 biggest transfers:")
for i, tx in enumerate(top_tx, 1):
    print(f"  {i}. {tx['amount']:,.6f}  ETH   "
          f"{tx['src'][:6]}… → {tx['dst'][:6]}…  "
          f"{tx['block_timestamp']:%Y-%m-%d}")

print("\nTop-5 busiest addresses (degree):")
for i, (addr, deg) in enumerate(top_degree, 1):
    print(f"  {i}. {addr[:6]}…   {deg:,} edges")

print("\nEverything looks readable ✔")


=====  GNN DATASET SUMMARY  =====

Accounts (nodes):        1,119,783
  ├─ labeled:            9,816
  ├─   └─ fraud (1):     2,179
  └─ unlabeled (NULL):   1,109,967

Transactions (edges):    85,544,758
  Time span:             2023-01-01  →  2025-04-30
  Amount, ETH:
    min  = 0.000000
    max  = 550,000.000000
    mean = 6.689520
    med. ≈ 0.100000

Top-5 biggest transfers:
  1. 550,000.000000  ETH   0xda9d… → 0x267b…  2024-05-30
  2. 448,219.680000  ETH   0xda9d… → 0x267b…  2025-01-31
  3. 207,817.164990  ETH   0x7713… → 0x742d…  2023-03-22
  4. 200,000.000000  ETH   0xda9d… → 0x267b…  2024-02-14
  5. 147,677.780000  ETH   0x267b… → 0x109b…  2025-02-05

Top-5 busiest addresses (degree):
  1. 0x3fc9…   12,794,579 edges
  2. 0xef1c…   3,138,339 edges
  3. 0x974c…   2,597,722 edges
  4. 0x7a25…   2,587,135 edges
  5. 0x267b…   2,453,646 edges

Everything looks readable ✔


In [1]:
import pandas, sys, pathlib, platform
print("pandas version:", pandas.__version__)
print("pandas path:   ", pathlib.Path(pandas.__file__).parent)
print("python:", platform.python_version())


pandas version: 2.2.3
pandas path:    d:\Fraud\.venv\Lib\site-packages\pandas
python: 3.12.0


# Etherscan

In [ ]:
import time
import requests
import pandas as pd

API_KEY = ""
BASE_URL = "https://api.etherscan.io/api"

def fetch_txs(address, page=1, offset=10000):
    """Загружает транзакции (обычные) для address."""
    params = {
        "module": "account",
        "action": "txlist",
        "address": address,
        "startblock": 0,
        "endblock": 99999999,
        "page": page,
        "offset": offset,
        "sort": "desc",
        "apikey": API_KEY
    }
    r = requests.get(BASE_URL, params=params)
    data = r.json()
    if data["status"] != "1":
        return []
    return data["result"]

def fetch_internal_txs(address, page=1, offset=10000):
    """Загружает внутренние (internal) транзакции."""
    params = {
        "module": "account",
        "action": "txlistinternal",
        "address": address,
        "startblock": 0,
        "endblock": 99999999,
        "page": page,
        "offset": offset,
        "sort": "desc",
        "apikey": API_KEY
    }
    r = requests.get(BASE_URL, params=params)
    data = r.json()
    if data["status"] != "1":
        return []
    return data["result"]

def get_all_txs(address):
    """Собирает обычные + internal, пагинируя до пустой страницы."""
    all_txs = []
    for fetcher in (fetch_txs, fetch_internal_txs):
        page = 1
        while True:
            txs = fetcher(address, page=page)
            if not txs:
                break
            all_txs.extend(txs)
            page += 1
            time.sleep(0.2)   # чтобы не превысить 5 rps
    return all_txs

# 1) Читаем LSTM-адреса
lstm = pd.read_csv("./data/transaction_dataset.csv", usecols=["Address","FLAG"])
seed_addrs = set(lstm["Address"].tolist())

# 2) BFS-обход глубины 2
visited = set(seed_addrs)
frontier = set(seed_addrs)
edges = []   # тут будем собирать ребра

for depth in range(2):
    next_frontier = set()
    for addr in frontier:
        txs = get_all_txs(addr)
        for tx in txs:
            src = tx.get("from")
            dst = tx.get("to")
            amt = tx.get("value")
            ts  = int(tx.get("block_timestamp", 0))
            edges.append((src, dst, amt, ts))
            # запоминаем нового соседа
            for nbr in (src, dst):
                if nbr not in visited:
                    visited.add(nbr)
                    next_frontier.add(nbr)
    frontier = next_frontier

# 3) Сохраняем transaction.csv
tx_df = pd.DataFrame(edges, columns=["src","dst","amount","block_timestamp"])
tx_df.to_csv("./data/transaction.csv", index=False)

# 4) Сбор account.csv
all_nodes = pd.Series(list(tx_df["src"]) + list(tx_df["dst"]), name="id")
all_nodes = all_nodes.drop_duplicates().to_frame()
# маппим метки: LSTM→FLAG, новые узлы = -1
label_map = dict(zip(lstm["Address"], lstm["FLAG"]))
all_nodes["label"] = all_nodes["id"].map(label_map).fillna(-1).astype(int)
all_nodes.to_csv("./data/account.csv", index=False)

print("Собрано:", tx_df.shape[0], "транзакций;", all_nodes.shape[0], "узлов.")